# CogFlow Phase 1: Foundation Setup

This notebook tests and validates the Phase 1 foundation components of CogFlow:
1. **Configuration Management** (`config.py`)
2. **Core Types** (`base/types.py`) 
3. **Prompt Templates** (`prompts/cot_templates.py`)

## Overview
CogFlow is a modular reasoning framework supporting:
- Chain-of-thought reasoning
- ReAct (Reasoning + Acting) patterns
- Self-consistency and reranking
- Both LLM-powered and algorithmic approaches

## Section 1: Install and Verify Dependencies

Install required packages and verify they import correctly.

In [ ]:
# Install required dependencies
import subprocess
import sys

packages = [
    "pydantic>=2.0.0",
    "pydantic-settings>=2.0.0",
    "python-dotenv>=1.0.0",
    "openai>=1.0.0",
    "anthropic>=0.18.0",
    "httpx>=0.26.0",
    "tenacity>=8.2.0",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All packages installed successfully!")

In [ ]:
# Verify imports
import pydantic
from pydantic import BaseModel, Field
from pydantic_settings import BaseSettings
from dotenv import load_dotenv
from typing import Optional, Dict, List, Any, Union
from datetime import datetime
from enum import Enum
import os

print(f"✅ Pydantic version: {pydantic.__version__}")
print("✅ All core imports successful!")

In [ ]:
# Add cogflow to path for imports
import sys
from pathlib import Path

# Navigate to cogflow root
cogflow_root = Path.cwd().parent
if str(cogflow_root) not in sys.path:
    sys.path.insert(0, str(cogflow_root.parent))  # Add CogFlow folder
    
print(f"✅ Added to path: {cogflow_root.parent}")

## Section 2: Read and Parse README File

Load and display the README.md to understand project requirements.

In [ ]:
# Read README.md file
readme_path = Path.cwd().parent.parent / "readme.md"
print(f"Looking for README at: {readme_path}")

if readme_path.exists():
    with open(readme_path, 'r', encoding='utf-8') as f:
        readme_content = f.read()
    print(f"✅ README loaded successfully ({len(readme_content)} characters)")
    print("\n" + "="*60)
    print("README Preview (first 2000 chars):")
    print("="*60)
    print(readme_content[:2000])
else:
    print(f"❌ README not found at {readme_path}")

## Section 3: Extract Requirements from README

Key components identified from README:
- **Reasoner**: Chain-of-Thought generator
- **AgentController**: ReAct-style loop
- **Reranker**: Candidate selection
- **CandidateGenerator**: Self-consistency sampling
- **OutputFormatter**: Response formatting

Let's implement the core types first.

## Section 4: Implement Core Types

Define the fundamental data structures for CogFlow.

In [ ]:
# Core Type Definitions for CogFlow
# These are the fundamental data structures used throughout the framework

class MessageRole(str, Enum):
    """Role of a message in conversation."""
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"


class Message(BaseModel):
    """A message in a conversation."""
    role: MessageRole = Field(description="Role of the message sender")
    content: str = Field(description="Content of the message")
    name: Optional[str] = Field(default=None, description="Optional name")
    tool_call_id: Optional[str] = Field(default=None, description="ID for tool response")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata")
    timestamp: datetime = Field(default_factory=datetime.now, description="Message timestamp")
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for API calls."""
        result = {"role": self.role.value, "content": self.content}
        if self.name:
            result["name"] = self.name
        if self.tool_call_id:
            result["tool_call_id"] = self.tool_call_id
        return result


class ReasoningStep(BaseModel):
    """
    A single step in the reasoning process.
    Represents one iteration of the Thought → Action → Observation cycle.
    """
    step_number: int = Field(default=0, description="Step sequence number")
    thought: str = Field(description="The reasoning thought")
    action: Optional[str] = Field(default=None, description="Action to take (tool name)")
    action_input: Optional[Any] = Field(default=None, description="Input for the action")
    observation: Optional[str] = Field(default=None, description="Result of the action")
    timestamp: datetime = Field(default_factory=datetime.now, description="Step timestamp")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata")
    
    def is_complete(self) -> bool:
        """Check if this step has observation (completed)."""
        return self.observation is not None
    
    def to_string(self) -> str:
        """Format step as readable string."""
        parts = [f"Thought: {self.thought}"]
        if self.action:
            parts.append(f"Action: {self.action}")
            if self.action_input:
                parts.append(f"Action Input: {self.action_input}")
        if self.observation:
            parts.append(f"Observation: {self.observation}")
        return "\n".join(parts)


class ToolCall(BaseModel):
    """A request to invoke a tool."""
    id: str = Field(description="Unique identifier for this tool call")
    name: str = Field(description="Name of the tool to call")
    arguments: Dict[str, Any] = Field(default_factory=dict, description="Arguments for the tool")


class AgentAction(BaseModel):
    """
    Represents an action to be taken by an agent.
    Returned when the agent decides to use a tool.
    """
    tool: str = Field(description="Name of the tool to use")
    tool_input: Any = Field(description="Input to pass to the tool")
    log: str = Field(default="", description="Log of agent's reasoning")
    tool_call_id: Optional[str] = Field(default=None, description="Unique ID for tracking")
    
    def to_reasoning_step(self) -> ReasoningStep:
        """Convert to a ReasoningStep."""
        return ReasoningStep(
            thought=self.log,
            action=self.tool,
            action_input=self.tool_input
        )


class AgentFinish(BaseModel):
    """
    Represents the final output of an agent.
    Returned when the agent decides it has completed the task.
    """
    return_values: Dict[str, Any] = Field(description="Dictionary of return values")
    log: str = Field(default="", description="Log of agent's final reasoning")
    
    @property
    def output(self) -> str:
        """Get the main output value."""
        return self.return_values.get("output", str(self.return_values))


class Candidate(BaseModel):
    """
    A candidate response with metadata.
    Used for self-consistency and reranking.
    """
    content: str = Field(description="The candidate response content")
    reasoning_trace: List[ReasoningStep] = Field(
        default_factory=list,
        description="List of reasoning steps"
    )
    confidence: float = Field(default=0.0, ge=0.0, le=1.0, description="Confidence score")
    score: Optional[float] = Field(default=None, description="Reranking score")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata")
    source: str = Field(default="unknown", description="Source of this candidate")
    timestamp: datetime = Field(default_factory=datetime.now, description="Generation timestamp")
    
    def get_reasoning_summary(self) -> str:
        """Get a summary of the reasoning trace."""
        if not self.reasoning_trace:
            return "No reasoning trace available."
        steps = [step.to_string() for step in self.reasoning_trace]
        return "\n---\n".join(steps)


# Type aliases
AgentOutput = Union[AgentAction, AgentFinish]

print("✅ Core types defined successfully!")

## Section 5: Test Core Types

Verify that all core types work correctly with sample data.

In [ ]:
# Test Message type
print("Testing Message type...")
msg = Message(
    role=MessageRole.USER,
    content="What is the weather in Paris?"
)
print(f"  Message: {msg.to_dict()}")
assert msg.role == MessageRole.USER
assert "weather" in msg.content
print("  ✅ Message test passed!\n")

# Test ReasoningStep type
print("Testing ReasoningStep type...")
step = ReasoningStep(
    step_number=1,
    thought="I need to search for weather information",
    action="search",
    action_input={"query": "weather in Paris"},
    observation="Paris: 15°C, partly cloudy"
)
print(f"  Step string:\n{step.to_string()}")
assert step.is_complete() == True
print("  ✅ ReasoningStep test passed!\n")

# Test AgentAction type
print("Testing AgentAction type...")
action = AgentAction(
    tool="calculator",
    tool_input="2 + 2",
    log="I need to calculate this sum"
)
reasoning_step = action.to_reasoning_step()
print(f"  Action tool: {action.tool}")
print(f"  Converted step thought: {reasoning_step.thought}")
assert action.tool == "calculator"
print("  ✅ AgentAction test passed!\n")

# Test AgentFinish type
print("Testing AgentFinish type...")
finish = AgentFinish(
    return_values={"output": "The answer is 42"},
    log="I have computed the final answer"
)
print(f"  Output: {finish.output}")
assert "42" in finish.output
print("  ✅ AgentFinish test passed!\n")

# Test Candidate type
print("Testing Candidate type...")
candidate = Candidate(
    content="Paris has mild weather with temperatures around 15°C",
    reasoning_trace=[step],
    confidence=0.85,
    source="gpt-4"
)
print(f"  Content: {candidate.content}")
print(f"  Confidence: {candidate.confidence}")
print(f"  Reasoning summary:\n{candidate.get_reasoning_summary()}")
assert candidate.confidence == 0.85
assert len(candidate.reasoning_trace) == 1
print("  ✅ Candidate test passed!\n")

print("="*50)
print("✅ ALL CORE TYPE TESTS PASSED!")
print("="*50)

## Section 6: Implement Configuration System

Create a flexible configuration management system for CogFlow.

In [ ]:
# Configuration System for CogFlow
from typing import Literal


class LLMConfig(BaseModel):
    """Configuration for LLM model parameters."""
    model_name: str = Field(default="gpt-4", description="Model name to use")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1024, gt=0)
    top_p: float = Field(default=1.0, ge=0.0, le=1.0)
    timeout: float = Field(default=60.0, gt=0)


class AgentConfig(BaseModel):
    """Configuration for agent behavior."""
    max_iterations: int = Field(default=10, gt=0)
    early_stopping: bool = Field(default=True)
    return_intermediate_steps: bool = Field(default=True)
    handle_parsing_errors: bool = Field(default=True)
    verbose: bool = Field(default=False)


class ReasonerConfig(BaseModel):
    """Configuration for reasoning module."""
    num_candidates: int = Field(default=1, ge=1)
    use_self_consistency: bool = Field(default=False)
    voting_method: Literal["majority", "weighted", "best_score"] = "majority"
    include_reasoning_trace: bool = Field(default=True)


class RerankerConfig(BaseModel):
    """Configuration for reranking module."""
    method: Literal["llm", "rule_based", "voting", "ensemble"] = "llm"
    top_k: int = Field(default=1, ge=1)
    score_threshold: float = Field(default=0.0, ge=0.0, le=1.0)


class CogFlowConfig(BaseSettings):
    """Main configuration for CogFlow framework."""
    
    # API Keys
    openai_api_key: Optional[str] = Field(default=None)
    anthropic_api_key: Optional[str] = Field(default=None)
    huggingface_api_key: Optional[str] = Field(default=None)
    
    # Provider settings
    default_provider: Literal["openai", "anthropic", "huggingface", "ollama"] = "openai"
    ollama_base_url: str = Field(default="http://localhost:11434")
    
    # Sub-configurations
    llm: LLMConfig = Field(default_factory=LLMConfig)
    agent: AgentConfig = Field(default_factory=AgentConfig)
    reasoner: ReasonerConfig = Field(default_factory=ReasonerConfig)
    reranker: RerankerConfig = Field(default_factory=RerankerConfig)
    
    # Logging
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR"] = "INFO"
    enable_tracing: bool = Field(default=True)
    
    class Config:
        env_prefix = "COGFLOW_"
        env_file = ".env"
        extra = "ignore"
    
    def get_api_key(self, provider: str) -> Optional[str]:
        """Get API key for a specific provider."""
        key_map = {
            "openai": self.openai_api_key,
            "anthropic": self.anthropic_api_key,
            "huggingface": self.huggingface_api_key,
        }
        return key_map.get(provider)
    
    def validate_provider(self, provider: str) -> bool:
        """Check if a provider is properly configured."""
        if provider == "ollama":
            return True
        return self.get_api_key(provider) is not None
    
    def to_dict(self) -> Dict[str, Any]:
        """Export configuration as dictionary (masking sensitive data)."""
        data = self.model_dump()
        for key in ["openai_api_key", "anthropic_api_key", "huggingface_api_key"]:
            if data.get(key):
                data[key] = "***" + data[key][-4:] if len(data[key]) > 4 else "***"
        return data


print("✅ Configuration classes defined!")

In [ ]:
# Test Configuration System
print("Testing Configuration System...")

# Create default configuration
config = CogFlowConfig()
print(f"\n1. Default Configuration:")
print(f"   Default provider: {config.default_provider}")
print(f"   LLM model: {config.llm.model_name}")
print(f"   LLM temperature: {config.llm.temperature}")
print(f"   Max iterations: {config.agent.max_iterations}")

# Create custom configuration
custom_config = CogFlowConfig(
    default_provider="anthropic",
    llm=LLMConfig(model_name="claude-3-opus", temperature=0.5, max_tokens=2048),
    agent=AgentConfig(max_iterations=15, verbose=True),
    reasoner=ReasonerConfig(num_candidates=5, use_self_consistency=True)
)
print(f"\n2. Custom Configuration:")
print(f"   Default provider: {custom_config.default_provider}")
print(f"   LLM model: {custom_config.llm.model_name}")
print(f"   LLM temperature: {custom_config.llm.temperature}")
print(f"   Num candidates: {custom_config.reasoner.num_candidates}")
print(f"   Self-consistency: {custom_config.reasoner.use_self_consistency}")

# Test validation
print(f"\n3. Provider Validation:")
print(f"   OpenAI configured: {config.validate_provider('openai')}")
print(f"   Ollama configured: {config.validate_provider('ollama')}")

# Test to_dict (with masked API keys)
print(f"\n4. Configuration export (masked):")
export = config.to_dict()
print(f"   Keys: {list(export.keys())[:5]}...")

print("\n✅ Configuration system test passed!")

## Section 7: Implement Prompt Templates

Create Chain-of-Thought (CoT) and ReAct prompt templates.

In [ ]:
# Import and test prompt templates
from prompts.cot_templates import (
    PromptTemplate,
    TemplateRegistry,
    get_template_registry,
    format_cot_prompt,
    format_react_prompt
)

# Get the template registry
registry = get_template_registry()
print("Available templates:", registry.list_templates())
print()

# Test Chain-of-Thought template
cot_prompt = format_cot_prompt(
    question="What is the capital of France and why is it important?",
    context="France is a country in Western Europe known for its rich history."
)
print("=== Chain-of-Thought Prompt ===")
print(cot_prompt[:500], "...")
print()

# Test ReAct template
tools = [
    {"name": "search", "description": "Search for information"},
    {"name": "calculator", "description": "Perform calculations"}
]
react_prompt = format_react_prompt(
    question="What is 25% of the population of Paris?",
    tools=tools,
    examples="Example: Thought: I need to find the population first."
)
print("=== ReAct Prompt ===")
print(react_prompt[:600], "...")
print()

# Test custom template
custom_template = PromptTemplate(
    name="custom_analysis",
    template="Analyze the following: {topic}\n\nConsider: {aspects}",
    variables=["topic", "aspects"],
    description="Custom analysis template"
)
registry.register(custom_template)
print("Registered custom template")
print("Updated templates:", registry.list_templates())

## Section 8: Integration Test - All Components Together

Verify all Phase 1 components work together seamlessly.

In [ ]:
# Integration Test: Simulate a complete reasoning pipeline flow

from base.types import (
    Message, MessageRole, ReasoningStep, AgentAction, 
    AgentFinish, Candidate, ToolDefinition, ConversationHistory
)
from config import CogFlowConfig, LLMProvider
from prompts.cot_templates import format_react_prompt

print("=== Integration Test: Complete Reasoning Pipeline ===\n")

# 1. Initialize configuration
config = CogFlowConfig()
print(f"1. Configuration loaded:")
print(f"   - Provider: {config.llm.provider}")
print(f"   - Max iterations: {config.agent.max_iterations}")
print()

# 2. Define available tools
tools = [
    ToolDefinition(
        name="search",
        description="Search the web for information",
        parameters={"query": {"type": "string", "description": "Search query"}}
    ),
    ToolDefinition(
        name="calculator",
        description="Perform mathematical calculations",
        parameters={"expression": {"type": "string", "description": "Math expression"}}
    )
]
print(f"2. Registered {len(tools)} tools: {[t.name for t in tools]}")
print()

# 3. Create conversation history
history = ConversationHistory()
history.add_message(Message(
    role=MessageRole.USER,
    content="What is the population of Tokyo multiplied by 2?"
))
print(f"3. Conversation started with {len(history.messages)} message(s)")
print()

# 4. Generate ReAct prompt
react_prompt = format_react_prompt(
    question=history.messages[0].content,
    tools=[{"name": t.name, "description": t.description} for t in tools]
)
print("4. Generated ReAct prompt (first 200 chars):")
print(f"   {react_prompt[:200]}...")
print()

# 5. Simulate reasoning steps
steps = []

# Step 1: Thought
step1 = ReasoningStep(
    thought="I need to find the population of Tokyo first, then multiply by 2.",
    action="search",
    action_input={"query": "Tokyo population 2024"},
    observation=None
)
steps.append(step1)
print(f"5. Reasoning Step 1: {step1.thought}")

# Step 2: Observation (simulated)
step1.observation = "Tokyo has a population of approximately 14 million people."
print(f"   Observation: {step1.observation}")

# Step 3: Action
step2 = ReasoningStep(
    thought="Now I need to multiply 14 million by 2.",
    action="calculator",
    action_input={"expression": "14000000 * 2"},
    observation="28000000"
)
steps.append(step2)
print(f"   Reasoning Step 2: {step2.thought}")
print(f"   Observation: {step2.observation}")
print()

# 6. Create final response
final_action = AgentFinish(
    return_values={"output": "The population of Tokyo (approximately 14 million) multiplied by 2 is 28 million."},
    reasoning_steps=steps,
    log="Successfully completed reasoning with 2 steps."
)
print(f"6. Final Answer: {final_action.return_values['output']}")
print()

# 7. Generate candidates (for reranking simulation)
candidates = [
    Candidate(content="28 million", score=0.95, metadata={"source": "calculation"}),
    Candidate(content="28,000,000 people", score=0.90, metadata={"source": "formatted"}),
    Candidate(content="Approximately 28 million", score=0.85, metadata={"source": "natural"})
]
print(f"7. Generated {len(candidates)} candidate responses:")
for i, c in enumerate(candidates, 1):
    print(f"   {i}. '{c.content}' (score: {c.score})")
print()

# 8. Verify all imports work
print("8. All Phase 1 components verified:")
print("   ✓ Core types (Message, ReasoningStep, etc.)")
print("   ✓ Configuration system (CogFlowConfig)")
print("   ✓ Prompt templates (CoT, ReAct)")
print("   ✓ Conversation history management")
print("   ✓ Tool definitions")
print("   ✓ Candidate generation for reranking")
print()
print("=== Phase 1 Foundation Setup Complete! ===")

## Section 9: Summary and Next Steps

### Phase 1 Completed Components:
1. **Core Types** (`base/types.py`): Message, ReasoningStep, AgentAction, AgentFinish, Candidate, ToolDefinition, ConversationHistory
2. **Configuration** (`config.py`): LLMConfig, AgentConfig, CandidateConfig, OutputConfig, RerankerConfig, CogFlowConfig
3. **Prompt Templates** (`prompts/cot_templates.py`): PromptTemplate, TemplateRegistry, CoT and ReAct templates

### Next Phase (Phase 2): LLM Providers & Tool Registry
- Create abstract LLM interface (`base/llm.py`)
- Implement OpenAI provider (`providers/openai_provider.py`)
- Implement Anthropic provider (`providers/anthropic_provider.py`)
- Create Tool Registry (`modules/tool_registry.py`)

### Files Created:
```
cogflow/
├── __init__.py
├── config.py
├── requirements.txt
├── base/
│   ├── __init__.py
│   └── types.py
├── prompts/
│   ├── __init__.py
│   └── cot_templates.py
└── notebooks/
    └── 01_foundation_setup.ipynb
```